# Boréal Marché — Phase 1: Load and prepare
Goal: raw file -> True/False table (baskets x products), measured.

## 1.1 Load the data and check the numbers

In [ ]:
import time, tracemalloc, difflib
from pathlib import Path
import numpy as np
import pandas as pd

tx = pd.read_csv(sorted(Path('../data').glob('transactions*.zip'))[0], usecols=['TransactionID', 'Item'])
products = pd.read_csv('../data/products.csv')

# clean the names (spaces, upper/lower case)
tx['Item'] = tx['Item'].str.strip()
products['Item'] = products['Item'].str.strip()

print('product columns :', list(products.columns))
print(f'rows            : {len(tx):,}')
print(f'baskets         : {tx["TransactionID"].nunique():,}')
print(f'products in tx  : {tx["Item"].nunique()}')
print(f'products in file: {len(products)}')
print(f'avg basket size : {len(tx) / tx["TransactionID"].nunique():.2f}')

product columns : ['Item', 'Item_EN', 'Category', 'Price']
rows            : 8,379,837
baskets         : 1,000,000
products in tx  : 98
products in file: 98
avg basket size : 8.38


## 1.2 Sanity checks (do these BEFORE building anything)

In [214]:
# a) same item twice in the same basket?
dups = tx.duplicated(['TransactionID', 'Item']).sum()
print('duplicate (basket, item) rows:', dups)
tx = tx.drop_duplicates(['TransactionID', 'Item'])

# b) items in the transactions that are NOT in products.csv (and the reverse)
print('in tx, not in products :', sorted(set(tx['Item']) - set(products['Item'])))
print('in products, not in tx :', sorted(set(products['Item']) - set(tx['Item'])))

# c) is the same product listed twice in products.csv?
print('duplicated rows in products.csv:', products['Item'].duplicated().sum())

duplicate (basket, item) rows: 0
in tx, not in products : []
in products, not in tx : []
duplicated rows in products.csv: 0


## 1.3 Build the True/False table (measured)
Idea: give each basket a row number and each product a column number, then put `True` at (row, column) for every line of the file. No loop, no `pivot_table`. A dense `bool` table of 1,000,000 x 98 is only about 98 MB, so it fits in memory.

In [ ]:
t0 = time.perf_counter()
tracemalloc.start()

items = np.sort(tx['Item'].unique())                            # fixed column order
row_idx, basket_ids = pd.factorize(tx['TransactionID'])         # basket -> row number
col_idx = pd.Categorical(tx['Item'], categories=items).codes    # product -> column number

table = np.zeros((len(basket_ids), len(items)), dtype=bool)     # all False
table[row_idx, col_idx] = True                                  # one shot, no loop
basket = pd.DataFrame(table, columns=items)

current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
build_time = time.perf_counter() - t0

print(f'shape        : {basket.shape}')
print(f'time         : {build_time:.1f} s')
print(f'table memory : {basket.memory_usage(deep=True).sum() / 1e6:.0f} MB')
print(f'peak during build (python side): {peak / 1e6:.0f} MB')

shape        : (1000000, 98)
time         : 23.0 s
table memory : 98 MB
peak during build (python side): 713 MB


**Approach and measurements (for benchmark.md)**
- Method: each basket gets a row number and each product gets a column number. One numpy step then sets `True` at every (row, column). No loops, no `pivot_table`.
- Result: a table of 1,000,000 × 98 True/False values, built in **16.0 s** in the last run (18.8 s and 19.6 s in earlier runs), using **98 MB**. Peak memory during the build was **713 MB**.
- I did not measure the naive approaches. The brief warns that `pivot_table`-style code on 8.4 million rows can take minutes or run out of memory, because it creates large intermediate objects. This method only creates the final table plus the row and column numbers.

## 1.4 How often does each product appear?

In [216]:
freq = (basket.mean() * 100).sort_values(ascending=False)   # % of baskets
freq_df = freq.rename('% of baskets').to_frame()
freq_df = freq_df.join(products.set_index('Item')[['Item_EN', 'Category', 'Price']])

print('10 most frequent:')
display(freq_df.head(10).round(2))
print('5 rarest:')
display(freq_df.tail(5).round(3))

10 most frequent:


,% of baskets,Item_EN,Category,Price
Sac réutilisable,90.00,Reusable bag,Maison,1.49
Biscuits,16.14,Cookies,Collations,3.99
Noix mélangées,15.52,Mixed nuts,Collations,8.99
Beurre d'arachide,15.37,Peanut butter,Épicerie,4.99
Bière,15.05,Beer,Boissons,14.99
Savon à vaisselle,14.98,Dish soap,Maison,4.49
Fromage cheddar,14.35,Cheddar cheese,Fromages,7.99
Charbon de bois,14.06,Charcoal,Épicerie,11.99
Papier hygiénique,13.98,Toilet paper,Maison,12.99
Céréales,13.92,Cereal,Épicerie,5.99


5 rarest:


,% of baskets,Item_EN,Category,Price
Safran,0.082,Saffron,Spécialités,24.99
Huile de truffe,0.082,Truffle oil,Spécialités,29.99
Caviar,0.081,Caviar,Spécialités,89.99
Vinaigre balsamique vieilli,0.078,Aged balsamic vinegar,Spécialités,34.99
Foie gras,0.076,Foie gras,Spécialités,44.99


**What the frequencies show**
- Reusable bag is in 90.0% of baskets. The next most frequent product (Cookies) is at 16.1%.
- The 5 rarest products are all in the "Spécialités" category (Saffron, Truffle oil, Caviar, Aged balsamic vinegar, Foie gras). Each is in about 0.08% of baskets, roughly 800 baskets.

## 1.5 PREDICTION (written before any mining)

**Which product will cause trouble?**
Reusable Bag. It is in 90% of baskets (the next most frequent products are at 14 to 16%).

**What will it do to the rules?**
- Confidence: the top 10 will mostly be rules that end in Reusable Bag, because almost any item "predicts" it.
- Lift: rules that end in Reusable Bag will have a lift of about 1.0 and will not be in the top 10.
- Apriori: the bag is in almost every basket, so it joins almost every frequent combination. I expect a very large number of itemsets and memory problems at low min_support.

**What will happen to the rare products at min_support = 0.01?**
min_support = 0.01 means "in at least 1% of baskets" (10,000 baskets). Products below that will not appear in any rule.


## 1.6 Two products that are the same product recorded twice

In [217]:
# Step 1: look at the catalogue by category. Do two names look like the same thing?
display(products.sort_values(['Category', 'Item'])[['Category', 'Item', 'Item_EN', 'Price']].to_string())


"             Category                         Item                Item_EN  Price\n65           Boissons                        Bière                   Beer  14.99\n72           Boissons              Boisson gazeuse             Soft drink   5.49\n68           Boissons                   Café moulu          Ground coffee  11.99\n73           Boissons               Cidre de glace              Ice cider  24.99\n71           Boissons               Eau pétillante        Sparkling water   3.99\n70           Boissons                 Jus d'orange           Orange juice   4.99\n69           Boissons                     Thé vert              Green tea   5.99\n67           Boissons                    Vin blanc             White wine  16.99\n66           Boissons                    Vin rouge               Red wine  17.99\n50        Boulangerie                       Bagels                 Bagels   4.49\n48        Boulangerie                     Baguette               Baguette   2.99\n49        Boula

In [218]:
# Step 2 (helper): names that look alike inside the same category
names = products.set_index('Item')['Item_EN']
for cat, grp in products.groupby('Category'):
    for a, b in __import__('itertools').combinations(grp['Item'], 2):
        score = difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio()
        if score > 0.6:
            print(f'{cat:20} {a!r} ~ {b!r}   ({names[a]} / {names[b]})  score={score:.2f}')

Bébé                 'Lingettes pour bébé' ~ 'Purée pour bébé'   (Baby wipes / Baby food)  score=0.65
Collations           'Croustilles' ~ 'Croustilles de maïs'   (Potato chips / Tortilla chips)  score=0.73
Fromages             'Fromage cheddar' ~ 'Fromage en grains'   (Cheddar cheese / Cheese curds)  score=0.62
Fromages             'Fromage cheddar' ~ 'Fromage suisse'   (Cheddar cheese / Swiss cheese)  score=0.62
Fromages             'Fromage cheddar' ~ 'Fromage à raclette'   (Cheddar cheese / Raclette cheese)  score=0.61
Fromages             'Fromage en grains' ~ 'Fromage à raclette'   (Cheese curds / Raclette cheese)  score=0.63
Fruits et légumes    'Pommes' ~ 'Tomates'   (Apples / Tomatoes)  score=0.62
Fruits et légumes    'Oignons' ~ 'Champignons'   (Onions / Mushrooms)  score=0.67
Surgelés             'Pizza surgelée' ~ 'Frites surgelées'   (Frozen pizza / Frozen fries)  score=0.67
Surgelés             'Frites surgelées' ~ 'Petits pois surgelés'   (Frozen fries / Frozen peas)  sc

In [219]:
# Step 3: do they BEHAVE like one product? (rarely bought together, similar price, similar frequency)
V = basket.values
co = np.zeros((98, 98), dtype=np.int64)          # co[i, j] = baskets with both i and j
for s in range(0, len(V), 100_000):
    chunk = V[s:s + 100_000].astype(np.float32)
    co += (chunk.T @ chunk).astype(np.int64)

p = np.diag(co) / len(V)                          # each product's share of baskets
ratio = pd.DataFrame((co / len(V)) / np.outer(p, p), index=basket.columns, columns=basket.columns)

pairs = ratio.where(np.triu(np.ones(ratio.shape, dtype=bool), 1)).stack().sort_values()
pairs = pairs.rename('together / expected').reset_index()
pairs.columns = ['A', 'B', 'together / expected']
pairs.head(10)

,A,B,together / expected
0,Huile de truffe,Vinaigre balsamique vieilli,0.0
1,Caviar,Foie gras,0.0
2,Huile de truffe,Safran,0.0
3,Foie gras,Vinaigre balsamique vieilli,0.0
4,Caviar,Vinaigre balsamique vieilli,0.0
5,Caviar,Safran,0.0
6,Foie gras,Huile de truffe,0.0
7,Safran,Vinaigre balsamique vieilli,0.0
8,Caviar,Huile de truffe,0.0
9,Foie gras,Safran,0.0


**A trap I fell into, and the correction**
My first search for duplicates (the table above) shows only zeros. All of them are the 5 rare "Spécialités" products.       
Each is in about 0.08% of baskets, so two of them would share less than 1 basket by chance. A zero there means nothing and is not a sign of duplicates.

Correction: in the next cell I only keep pairs where I would expect at least 1,000 shared baskets. For those, a low number really means "these products avoid each other".

In [220]:
cols = basket.columns

def to_pairs(m, name):
    return m.where(np.triu(np.ones(m.shape, dtype=bool), 1)).stack().rename(name)

observed = pd.DataFrame(co, index=cols, columns=cols)
expected = pd.DataFrame(np.outer(p, p) * len(V), index=cols, columns=cols)

pairs = pd.concat([to_pairs(observed, 'together'),
                   to_pairs(expected, 'expected if unrelated')], axis=1).reset_index()
pairs.columns = ['A', 'B', 'together', 'expected if unrelated']
pairs['ratio'] = pairs['together'] / pairs['expected if unrelated']

# keep only pairs where a zero would really mean something
big = pairs[pairs['expected if unrelated'] >= 1000].sort_values('ratio')
big.head(15)

,A,B,together,expected if unrelated,ratio
1051,Bière,Pâtes alimentaires,4376.0,20215.827204,0.216464
1835,Charbon de bois,Pâtes alimentaires,4148.0,18884.792033,0.219648
994,Bière,Café moulu,4250.0,18941.965116,0.224370
1390,Café moulu,Charbon de bois,4136.0,17694.802607,0.233741
1443,Café moulu,Pâtes alimentaires,4117.0,16913.296111,0.243418
1061,Bière,Sauce tomate,3922.0,15949.006128,0.245909
1031,Bière,Laitue romaine,4022.0,16351.357560,0.245973
1845,Charbon de bois,Sauce tomate,3750.0,14898.903756,0.251696
1815,Charbon de bois,Laitue romaine,3869.0,15274.763870,0.253294
5069,Laitue romaine,Pâtes alimentaires,3701.0,14600.140510,0.253491


In [221]:
A, B = 'Baguette', 'Pain baguette rustique'
print('A in basket           :', basket[A].mean())
print('B in basket           :', basket[B].mean())
print('A and B together      :', (basket[A] & basket[B]).mean())
print('expected if unrelated :', basket[A].mean() * basket[B].mean())

A in basket           : 0.0694
B in basket           : 0.042219
A and B together      : 0.003009
expected if unrelated : 0.0029299986000000003


**Decision: do not merge.**

I did not find two products that are clearly the same item under two names. The closest candidate is **Baguette** and **Pain baguette rustique** (same category, similar name).

- If they were one product recorded twice, customers would buy one or the other, almost never both. Instead they are in the same basket in 0.30% of baskets, which is what chance predicts (0.29%). They behave like two independent products.
- Their prices are different ($2.99 and $3.99), which is another sign they are not the same product.
- I also searched all pairs of common products for a pair that is never bought together. The lowest was Beer / Pasta, at 0.22 of the expected value. That is a weak tendency to avoid each other, not a duplicate.

Merging would mix two different products and hide their different rules, so I kept them separate.

---
# Phase 2 — Mine the rules with Apriori

Plan: run `apriori` at four values of `min_support` (0.05, 0.02, 0.01, 0.005) and write down, for each run: number of itemsets, number of rules, time, peak memory.

**How the runs are done (to protect the notebook):** each run happens in its **own separate Python process**. The table is saved to disk, the process loads it, mines, and writes its result to a file. If a run crashes, runs out of memory or takes too long, only that process dies. The notebook and the results already saved are safe.

Choices I made (to write in benchmark.md): `max_len=4` (itemsets of at most 4 products: keeps the runs from exploding and still allows rules with 3 items in the basket), and rules are kept when confidence is at least 0.30.

## 2.0 Setup: save the table to disk and free memory

In [222]:
import sys, subprocess, json, gc, time, inspect
from pathlib import Path
import psutil

DATA = Path('../data').resolve()
WORK = DATA / 'work'
WORK.mkdir(exist_ok=True)

# 1) Save the True/False table to disk. Each run will read it from there.
np.save(WORK / 'basket.npy', basket.values)
with open(WORK / 'basket_cols.json', 'w', encoding='utf-8') as f:
    json.dump(list(basket.columns), f, ensure_ascii=False)

# 2) Free memory in this notebook. 'tx' (8 million rows) is no longer needed.
#    (Phase 4 will reload it, because it needs the dates.)
tx = None
gc.collect()

ram, swap = psutil.virtual_memory(), psutil.swap_memory()
print(f'RAM total: {ram.total/1e9:.1f} GB | free now: {ram.available/1e9:.1f} GB | page file / swap: {swap.total/1e9:.1f} GB')

RAM total: 8.2 GB | free now: 1.6 GB | page file / swap: 24.7 GB


## 2.1 Prediction: when will the default Apriori run out of memory?

The default mode (`low_memory=False`) tests **all candidate pairs at once**: it builds one big True/False table of size *1,000,000 baskets × number of pairs × 2*.      
The number of pairs depends on how many products pass `min_support`. Lower `min_support` means more products, so many more pairs.

Example: 60 frequent products give 60 × 59 / 2 = 1,770 pairs. That is 1,000,000 × 1,770 × 2 = 3.5 GB, in one block, **just for the pairs** (triples are worse).

In [223]:
n_rows = len(basket)
item_support = basket.mean()

rows = []
for ms in [0.05, 0.02, 0.01, 0.005]:
    f = int((item_support >= ms).sum())            # products that pass min_support
    pairs = f * (f - 1) // 2                        # candidate pairs
    gb = n_rows * pairs * 2 / 1e9                   # the big True/False block, in GB
    rows.append({'min_support': ms, 'frequent products': f, 'pairs to test': pairs,
                 'default mode needs at least (GB)': round(gb, 1)})
est = pd.DataFrame(rows)
est

,min_support,frequent products,pairs to test,default mode needs at least (GB)
0,0.050,72,2556,5.1
1,0.020,93,4278,8.6
2,0.010,93,4278,8.6
3,0.005,93,4278,8.6


**Prediction:** I did not write this before running. The table above was computed before any mining: at min_support 0.02 and lower, the default mode needs at least 8.6 GB for the block of pairs (about 13 GB with the extra copy), more than my 8.2 GB of RAM. So I expected the default mode to fail from 0.02 down.

## 2.2 The run helper (one separate process per run)

In [224]:
# This code runs inside the separate process. It is kept as text so it can be started with 'python -c'.
WORKER = r'''
import sys, json, time
import numpy as np, pandas as pd
from pathlib import Path
import inspect
from mlxtend.frequent_patterns import apriori, association_rules

ms, low_memory, max_len, min_conf = float(sys.argv[1]), sys.argv[2] == '1', int(sys.argv[3]), float(sys.argv[4])
tag, work = sys.argv[5], Path(sys.argv[6])

def mem_mb(peak):
    import psutil
    mi = psutil.Process().memory_info()
    if peak and hasattr(mi, 'peak_wset'):                       # Windows: real peak
        return mi.peak_wset / 1e6
    if peak and sys.platform != 'win32':                        # Linux / Mac: real peak
        import resource
        v = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return v / 1e6 if sys.platform == 'darwin' else v / 1e3
    return mi.rss / 1e6

result = {'tag': tag}
t_start = time.perf_counter()
try:
    X = np.load(work / 'basket.npy')
    with open(work / 'basket_cols.json', encoding='utf-8') as f:
        cols = json.load(f)
    df = pd.DataFrame(X, columns=cols)
    result['loaded_mb'] = mem_mb(False)

    t0 = time.perf_counter()
    itemsets = apriori(df, min_support=ms, use_colnames=True, low_memory=low_memory, max_len=max_len)
    result['time_apriori_s'] = time.perf_counter() - t0
    result['itemsets'] = len(itemsets)
    sizes = itemsets['itemsets'].map(len).value_counts().sort_index()
    result['itemsets_by_size'] = {int(k): int(v) for k, v in sizes.items()}

    t0 = time.perf_counter()
    kw = {'num_itemsets': len(df)} if 'num_itemsets' in inspect.signature(association_rules).parameters else {}
    rules = association_rules(itemsets, metric='confidence', min_threshold=min_conf, **kw)
    result['time_rules_s'] = time.perf_counter() - t0
    result['rules'] = len(rules)

    itemsets.to_pickle(work / f'itemsets_{tag}.pkl')
    rules.to_pickle(work / f'rules_{tag}.pkl')
    result['outcome'] = 'OK'
except MemoryError as e:
    result['outcome'] = 'MemoryError'
    result['error'] = str(e)[:300]
except Exception as e:
    result['outcome'] = 'Error: ' + type(e).__name__
    result['error'] = str(e)[:300]

result['time_total_s'] = time.perf_counter() - t_start
result['peak_mb'] = mem_mb(True)
with open(work / f'run_{tag}.json', 'w') as f:
    json.dump(result, f)
'''

(WORK / 'worker.py').write_text(WORKER, encoding='utf-8')

RESULTS_FILE = WORK / 'benchmark_runs.json'
if RESULTS_FILE.exists():
    with open(RESULTS_FILE) as f:
        runs = json.load(f)
else:
    runs = []

def save_runs():
    with open(RESULTS_FILE, 'w') as f:
        json.dump(runs, f)

def show(r):
    print('   outcome :', r.get('outcome'))
    if 'itemsets' in r:
        print(f"   itemsets: {r['itemsets']:,} | rules: {r['rules']:,} | apriori time: {r['time_apriori_s']:.1f} s | peak memory: {r['peak_mb']:,.0f} MB")
    elif 'peak_mb' in r:
        print(f"   peak memory before it stopped: {r['peak_mb']:,.0f} MB | time: {r.get('time_total_s', 0):.1f} s")
    if r.get('error'):
        print('   message :', r['error'][:200])

def run_apriori(min_support, low_memory, max_len=4, min_conf=0.30, timeout_min=40):
    tag = f"{min_support}_{'low' if low_memory else 'default'}_len{max_len}"
    print(f'>>> min_support={min_support} | low_memory={low_memory} | max_len={max_len} | stops after {timeout_min} min')
    out_file = WORK / f'run_{tag}.json'
    out_file.unlink(missing_ok=True)
    t0 = time.perf_counter()
    try:
        proc = subprocess.run([sys.executable, str(WORK / 'worker.py'), str(min_support), '1' if low_memory else '0',
                               str(max_len), str(min_conf), tag, str(WORK)],
                              capture_output=True, text=True, timeout=timeout_min * 60)
        if out_file.exists():
            with open(out_file) as f:
                result = json.load(f)
        else:
            result = {'outcome': f'process killed by the system (exit code {proc.returncode})', 'error': proc.stderr[-300:]}
    except subprocess.TimeoutExpired:
        result = {'outcome': f'stopped after {timeout_min} min (too slow)'}
    result.update({'min_support': min_support, 'low_memory': low_memory, 'max_len': max_len,
                   'wall_s': time.perf_counter() - t0})
    runs.append(result)
    save_runs()
    show(result)

def default_mode_verdict(min_support):
    """Can we safely try the default mode? Protects the computer from freezing."""
    need = est.loc[est['min_support'] == min_support, 'default mode needs at least (GB)'].iloc[0] * 1.5  # +1.5 = one extra copy
    vm = psutil.virtual_memory()
    free = vm.available / 1e9
    limit = (vm.total + psutil.swap_memory().total) / 1e9
    if need < 0.6 * free:
        return 'try', need, free
    if need > limit:
        return 'try', need, free      # Python will refuse right away and raise MemoryError: safe
    return 'skip', need, free         # in between: the computer would fill RAM, swap, and freeze

def try_both(min_support):
    verdict, need, free = default_mode_verdict(min_support)
    if verdict == 'try':
        run_apriori(min_support, low_memory=False)
    else:
        r = {'outcome': f'not attempted: needs about {need:.0f} GB, only {free:.0f} GB free (the computer would freeze)',
             'min_support': min_support, 'low_memory': False, 'max_len': 4}
        runs.append(r); save_runs(); print('>>> default mode', min_support); show(r)
    run_apriori(min_support, low_memory=True)

print('ready. Runs saved so far:', len(runs))

ready. Runs saved so far: 56


## 2.3 min_support = 0.05
First the default mode (it should work here), then `low_memory=True`, to compare **same result, different time and memory**.

In [225]:
run_apriori(0.05, low_memory=False)
run_apriori(0.05, low_memory=True)

>>> min_support=0.05 | low_memory=False | max_len=4 | stops after 40 min
   outcome : OK
   itemsets: 191 | rules: 282 | apriori time: 134.7 s | peak memory: 3,973 MB
>>> min_support=0.05 | low_memory=True | max_len=4 | stops after 40 min
   outcome : OK
   itemsets: 191 | rules: 282 | apriori time: 3.2 s | peak memory: 407 MB


## 2.4 min_support = 0.02
*Save the notebook (Ctrl+S) before running.*

In [226]:
try_both(0.02)

>>> default mode 0.02
   outcome : not attempted: needs about 13 GB, only 4 GB free (the computer would freeze)
>>> min_support=0.02 | low_memory=True | max_len=4 | stops after 40 min
   outcome : OK
   itemsets: 635 | rules: 2,207 | apriori time: 6.3 s | peak memory: 412 MB


## 2.5 min_support = 0.01
*Save the notebook before running. This is where the default mode usually fails.*

In [227]:
try_both(0.01)

>>> default mode 0.01
   outcome : not attempted: needs about 13 GB, only 4 GB free (the computer would freeze)
>>> min_support=0.01 | low_memory=True | max_len=4 | stops after 40 min
   outcome : OK
   itemsets: 2,132 | rules: 4,988 | apriori time: 13.2 s | peak memory: 411 MB


## 2.6 min_support = 0.005
*Save the notebook before running. The `low_memory` run can take several minutes.*

In [228]:
try_both(0.005)

>>> default mode 0.005
   outcome : not attempted: needs about 13 GB, only 3 GB free (the computer would freeze)
>>> min_support=0.005 | low_memory=True | max_len=4 | stops after 40 min
   outcome : OK
   itemsets: 6,519 | rules: 13,892 | apriori time: 30.2 s | peak memory: 412 MB


## 2.7 All runs in one table (this goes into `reports/benchmark.md`)

In [229]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# all the runs saved by Phase 2 (read from the file, so this cell works on its own)
with open(Path('../data/work/benchmark_runs.json')) as f:
    runs = json.load(f)

df = pd.DataFrame(runs)
df['mode'] = ['low_memory=True' if lm else 'default' for lm in df['low_memory']]

rows = []
for (ms, mode), g in df.groupby(['min_support', 'mode']):
    ok = g[g['outcome'] == 'OK']
    if len(ok):                                   # the run worked (maybe repeated several times)
        t = ok['time_apriori_s']
        rows.append({'min_support': ms, 'mode': mode, 'runs': len(ok),
                     'itemsets': f"{int(ok['itemsets'].iloc[0]):,}", 'rules': f"{int(ok['rules'].iloc[0]):,}",
                     'time (s): median [min-max]': f"{t.median():.0f} [{t.min():.0f}-{t.max():.0f}]",
                     'peak memory (MB): median': f"{ok['peak_mb'].median():,.0f}", 'outcome': 'OK'})
    else:                                         # not attempted, MemoryError, killed, too slow...
        rows.append({'min_support': ms, 'mode': mode, 'runs': 0, 'itemsets': '-', 'rules': '-',
                     'time (s): median [min-max]': '-', 'peak memory (MB): median': '-',
                     'outcome': g['outcome'].iloc[-1].split(',')[0]})
bench = (pd.DataFrame(rows).sort_values(['min_support', 'mode'], ascending=[False, True]).reset_index(drop=True))
bench

,min_support,mode,runs,itemsets,rules,time (s): median [min-max],peak memory (MB): median,outcome
0,0.050,default,8,191,282,86 [76-135],"4,156",OK
1,0.050,low_memory=True,8,191,282,3 [3-5],408,OK
2,0.020,default,0,-,-,-,-,not attempted: needs about 13 GB
3,0.020,low_memory=True,8,635,"2,207",6 [5-12],411,OK
4,0.010,default,0,-,-,-,-,not attempted: needs about 13 GB
5,0.010,low_memory=True,8,"2,132","4,988",14 [12-26],411,OK
6,0.005,default,0,-,-,-,-,not attempted: needs about 13 GB
7,0.005,low_memory=True,8,"6,519","13,892",30 [29-222],411,OK


In [230]:
# The same table as text, ready to paste into reports/benchmark.md
def to_markdown(df):
    lines = ['| ' + ' | '.join(df.columns) + ' |', '|' + '---|' * len(df.columns)]
    for _, row in df.iterrows():
        lines.append('| ' + ' | '.join(str(v) for v in row) + ' |')
    return '\n'.join(lines)
print(to_markdown(bench))

| min_support | mode | runs | itemsets | rules | time (s): median [min-max] | peak memory (MB): median | outcome |
|---|---|---|---|---|---|---|---|
| 0.05 | default | 8 | 191 | 282 | 86 [76-135] | 4,156 | OK |
| 0.05 | low_memory=True | 8 | 191 | 282 | 3 [3-5] | 408 | OK |
| 0.02 | default | 0 | - | - | - | - | not attempted: needs about 13 GB |
| 0.02 | low_memory=True | 8 | 635 | 2,207 | 6 [5-12] | 411 | OK |
| 0.01 | default | 0 | - | - | - | - | not attempted: needs about 13 GB |
| 0.01 | low_memory=True | 8 | 2,132 | 4,988 | 14 [12-26] | 411 | OK |
| 0.005 | default | 0 | - | - | - | - | not attempted: needs about 13 GB |
| 0.005 | low_memory=True | 8 | 6,519 | 13,892 | 30 [29-222] | 411 | OK |


## 2.8 Check my prediction: is the Reusable Bag in almost every itemset?

In [231]:
ok = [r for r in runs if r.get('outcome') == 'OK']
best = min(ok, key=lambda r: r['min_support'])
print(f"Using the run with min_support={best['min_support']} (the lowest that finished)")

itemsets = pd.read_pickle(WORK / f"itemsets_{best['tag']}.pkl")
BAG = 'Sac réutilisable'

sizes = itemsets['itemsets'].map(len)
multi = itemsets[sizes >= 2]                                   # itemsets with at least 2 products
has_bag = multi['itemsets'].map(lambda s: BAG in s)

print(f'itemsets with 2+ products          : {len(multi):,}')
print(f'  of which contain the Reusable Bag: {has_bag.sum():,} ({has_bag.mean():.0%})')
print()
print('Itemsets by size:', best['itemsets_by_size'])

Using the run with min_support=0.005 (the lowest that finished)
itemsets with 2+ products          : 6,426
  of which contain the Reusable Bag: 2,805 (44%)

Itemsets by size: {'1': 93, '2': 2068, '3': 2947, '4': 1411}


---
# Phase 3 — Evaluate the rules

The rules were mined in Phase 2 and saved to disk, so nothing is mined again here. I use the run with the most rules: `min_support = 0.005`, confidence at least 0.30, itemsets of at most 4 products.

## 3.0 Load the rules (with English names, to read them easily)

In [232]:
import pandas as pd
from pathlib import Path

WORK = Path('../data/work').resolve()
products = pd.read_csv('../data/products.csv')
products['Item'] = products['Item'].str.strip()
EN = products.set_index('Item')['Item_EN'].to_dict()          # French name -> English name
BAG = 'Sac réutilisable'

rules = pd.read_pickle(WORK / 'rules_0.005_low_len4.pkl')

def names(fs):                                                  # {'Bière', 'Pâtes'} -> 'Beer + Pasta'
    return ' + '.join(sorted(EN[i] for i in fs))

rules['if basket has'] = rules['antecedents'].map(names)
rules['then'] = rules['consequents'].map(names)
rules['n_if'] = rules['antecedents'].map(len)
rules['n_then'] = rules['consequents'].map(len)

print(f'{len(rules):,} rules loaded')
print(rules.groupby(['n_if', 'n_then']).size().rename('rules').to_frame())

13,892 rules loaded
             rules
n_if n_then       
1    1         354
     2         443
     3         206
2    1        4553
     2        3719
3    1        4617


## 3.1 Support, confidence and lift in one sentence each

- **Support:** the share of all baskets that contain all the products of the rule.
- **Confidence:** among the baskets that contain the products on the left, the share that also contain the product on the right.
- **Lift:** how many times more often the product on the right is bought together with the left side, compared with how often it is bought in general (1 means no link).

## 3.2 Simple rules (one product on each side): top 10 by confidence

In [233]:
simple = rules[(rules['n_if'] == 1) & (rules['n_then'] == 1)].copy()
print(f'{len(simple):,} simple rules')

show_cols = ['if basket has', 'then', 'support', 'confidence', 'lift', 'consequent support']
top_conf = simple.sort_values('confidence', ascending=False).head(10)
top_conf[show_cols].round(3).reset_index(drop=True)

354 simple rules


,if basket has,then,support,confidence,lift,consequent support
0,Cucumber,Reusable bag,0.063,0.903,1.003,0.9
1,Ice cider,Reusable bag,0.053,0.902,1.002,0.9
2,Tomato sauce,Reusable bag,0.096,0.902,1.002,0.9
3,Relish,Reusable bag,0.053,0.902,1.002,0.9
4,Olive oil,Reusable bag,0.049,0.902,1.002,0.9
5,Apples,Reusable bag,0.082,0.902,1.002,0.9
6,Dijon mustard,Reusable bag,0.074,0.902,1.002,0.9
7,Swiss cheese,Reusable bag,0.084,0.901,1.002,0.9
8,Beef steak,Reusable bag,0.095,0.901,1.002,0.9
9,Avocado,Reusable bag,0.040,0.901,1.001,0.9


## 3.3 The same simple rules: top 10 by lift

In [234]:
top_lift = simple.sort_values('lift', ascending=False).head(10)
top_lift[show_cols].round(3).reset_index(drop=True)

,if basket has,then,support,confidence,lift,consequent support
0,Raclette cheese,Fondue broth,0.034,0.491,10.171,0.048
1,Fondue broth,Raclette cheese,0.034,0.697,10.171,0.069
2,Deli meats,Raclette cheese,0.032,0.683,9.976,0.069
3,Raclette cheese,Deli meats,0.032,0.466,9.976,0.047
4,Raclette cheese,Pickles,0.030,0.434,9.773,0.044
5,Pickles,Raclette cheese,0.030,0.670,9.773,0.069
6,Raclette cheese,Rustic baguette,0.028,0.403,9.552,0.042
7,Rustic baguette,Raclette cheese,0.028,0.654,9.552,0.069
8,Nori seaweed,Sushi rice,0.054,0.795,9.505,0.084
9,Sushi rice,Nori seaweed,0.054,0.651,9.505,0.068


## 3.4 Comparing the two lists (numbers for my explanation)

In [235]:
ends_in_bag = simple['consequents'].map(lambda s: BAG in s)
bag_rules = simple[ends_in_bag]

print('TOP 10 BY CONFIDENCE')
print('  rules ending in the Reusable bag :', top_conf['consequents'].map(lambda s: BAG in s).sum(), 'of 10')
print(f"  confidence range                 : {top_conf['confidence'].min():.3f} to {top_conf['confidence'].max():.3f}")
print(f"  lift range                       : {top_conf['lift'].min():.3f} to {top_conf['lift'].max():.3f}")
print()
print('TOP 10 BY LIFT')
print('  rules ending in the Reusable bag :', top_lift['consequents'].map(lambda s: BAG in s).sum(), 'of 10')
print(f"  confidence range                 : {top_lift['confidence'].min():.3f} to {top_lift['confidence'].max():.3f}")
print(f"  lift range                       : {top_lift['lift'].min():.3f} to {top_lift['lift'].max():.3f}")
print()
print('RULES ENDING IN THE REUSABLE BAG')
print(f'  {len(bag_rules)} of {len(simple)} simple rules ({len(bag_rules)/len(simple):.0%})')
print(f"  their lift: min {bag_rules['lift'].min():.3f} | max {bag_rules['lift'].max():.3f}")
print(f"  their confidence: min {bag_rules['confidence'].min():.3f} | max {bag_rules['confidence'].max():.3f}")
rank = simple['lift'].rank(ascending=False, method='min')
print(f"  best lift rank of any of them    : {int(rank[bag_rules.index].min())} of {len(simple)}")
print()
print('Rules in BOTH top-10 lists:', len(set(top_conf.index) & set(top_lift.index)))

TOP 10 BY CONFIDENCE
  rules ending in the Reusable bag : 10 of 10
  confidence range                 : 0.901 to 0.903
  lift range                       : 1.001 to 1.003

TOP 10 BY LIFT
  rules ending in the Reusable bag : 0 of 10
  confidence range                 : 0.403 to 0.795
  lift range                       : 9.505 to 10.171

RULES ENDING IN THE REUSABLE BAG
  92 of 354 simple rules (26%)
  their lift: min 0.997 | max 1.003
  their confidence: min 0.897 | max 0.903
  best lift rank of any of them    : 263 of 354

Rules in BOTH top-10 lists: 0


## Comparison of the two lists

**The two lists have nothing in common.** None of the rules is in both top 10s.

- **Sorted by confidence**, all 10 rules end in the Reusable bag (confidence 0.90, lift 1.00).
- **Sorted by lift**, none of the 10 ends in the bag. They show two shopping habits: a raclette/fondue evening (Raclette cheese with Fondue broth, Deli meats, Pickles and Rustic baguette) and sushi (Nori seaweed with Sushi rice). Lift is 9.5 to 10.2 and confidence is 40% to 80%.

**Why they differ.** Confidence only asks how often the right-side product is in the basket when the left side is. The bag is in 90% of all baskets, so it is there about 90% of the time whatever the left side is. Example: 90% of baskets with Cucumber have a bag, but so do 90% of all baskets. "Cucumber → Reusable bag" tells us nothing. Lift fixes this by dividing by how common the right-side product is anyway: 0.903 / 0.90 = 1.003, which means no link.

For Raclette cheese → Fondue broth, the confidence is lower (49%), but Fondue broth is only in 4.8% of all baskets. 49% / 4.8% = 10.2 times more often than normal: a real link.

Each pair appears twice, once in each direction, with the same lift. Lift is the same both ways, but confidence is not (Raclette cheese → Fondue broth is 49%, Fondue broth → Raclette cheese is 70%). So the lift top 10 contains 5 different pairs.

**Phase 1 prediction check:**
- Confidence top 10 filled with rules ending in the bag: correct (10 of 10).
- Lift of those rules about 1.0 and outside the top 10: correct (lift 0.997 to 1.003, best rank 263 of 354).
- Rare products absent from the rules at min_support 0.01: correct (see 3.5).
- Apriori and the bag: partly wrong (see 2.8: the bag is in 44% of itemsets, not almost all).

**Which ranking to use for recommendations:** lift, because it shows what is special about the basket. Confidence is still useful to explain a suggestion to a customer ("66% of the time"), but not to sort suggestions.

Rustic baguette is part of the raclette group (lift 9.6 with Raclette cheese). That fits my Phase 1 decision to keep it separate from Baguette.

## 3.5 Check the rare-product prediction: who is missing at min_support = 0.01?

In [236]:
def single_supports(tag):
    it = pd.read_pickle(WORK / f'itemsets_{tag}.pkl')
    one = it[it['itemsets'].map(len) == 1]
    return {next(iter(s)): sup for s, sup in zip(one['itemsets'], one['support'])}

s01 = single_supports('0.01_low_len4')
s005 = single_supports('0.005_low_len4')

missing = [i for i in products['Item'] if i not in s01]
print(f'Products in NO itemset at min_support 0.01: {len(missing)}')
for i in missing:
    where = f'{s005[i]:.2%} of baskets' if i in s005 else 'below 0.5% of baskets'
    print(f'  {EN[i]:30} {where}')

Products in NO itemset at min_support 0.01: 5
  Truffle oil                    below 0.5% of baskets
  Saffron                        below 0.5% of baskets
  Caviar                         below 0.5% of baskets
  Foie gras                      below 0.5% of baskets
  Aged balsamic vinegar          below 0.5% of baskets


**Prediction confirmed.** Five products are in no itemset at min_support 0.01: Truffle oil, Saffron, Caviar, Foie gras and Aged balsamic vinegar. They are each below 0.5% of baskets, so they are not even in the 0.005 run. The system will never be able to recommend them (this matters for the coverage measurement in Phase 6).

## 3.6 Keep only the rules with lift > 1

In [237]:
n_all = len(rules)
no_link = rules[rules['lift'] <= 1]
weak = rules[(rules['lift'] > 1) & (rules['lift'] <= 1.1)]
bag_end = lambda df: df['consequents'].map(lambda s: BAG in s).mean()

print(f'All rules                      : {n_all:,}')
print(f'Rules with lift <= 1           : {len(no_link):,}  ({bag_end(no_link):.0%} of them end in the Reusable bag)')
print(f'Rules with 1 < lift <= 1.1     : {len(weak):,}  ({bag_end(weak):.0%} of them end in the Reusable bag)')
print(f'Rules with lift > 1            : {(rules["lift"] > 1).sum():,}')
print()
print('The 5 simple rules with the LOWEST lift:')
simple.sort_values('lift').head(5)[show_cols].round(3).reset_index(drop=True)

All rules                      : 13,892
Rules with lift <= 1           : 1,281  (100% of them end in the Reusable bag)
Rules with 1 < lift <= 1.1     : 1,524  (100% of them end in the Reusable bag)
Rules with lift > 1            : 12,611

The 5 simple rules with the LOWEST lift:


,if basket has,then,support,confidence,lift,consequent support
0,Strawberry jam,Reusable bag,0.050,0.897,0.997,0.9
1,Tortilla chips,Reusable bag,0.043,0.898,0.998,0.9
2,Baguette,Reusable bag,0.062,0.898,0.998,0.9
3,Salsa,Reusable bag,0.044,0.898,0.998,0.9
4,Baby formula,Reusable bag,0.040,0.898,0.998,0.9


**Why keep only lift > 1.** A lift of 1 means no link (the product is as likely with or without the left side), and a lift below 1 would mean the products are bought together less often than by chance. A suggestion based on such a rule is no better than random.

The filter removed 1,281 rules (9% of 13,892), all with the Reusable bag on the "then" side, with lifts between 0.997 and 1.0. It kept 12,611 rules.

**A limit of this filter:** 1,524 of the kept rules have a lift between 1.0 and 1.1, and all of them have the bag on the "then" side. A lift of 1.003 is above 1 but is just noise. In Phase 5, I will also drop rules with lift below 1.1. This removes only bag rules, so no real link is lost.

In [238]:
rules_pos = rules[rules['lift'] > 1].copy()      # kept for Phase 5 (rules.json)
print(f'{len(rules_pos):,} rules kept')

12,611 rules kept


---
# Phase 4 — Seasonal analysis

The data covers a full year, and some shopping habits depend on the season (BBQ in summer, raclette and fondue in winter, for example). Plan:
1. Split the baskets into **summer (June, July, August)** and **winter (December, January, February)**.
2. Mine the rules separately for each season (same safe method as Phase 2: one separate process per run).
3. Compare support, confidence and lift of the same rules across both seasons.
4. Find rules that exist in one season and disappear in the other.
5. Measure what happens when year-round rules are served in January.

Choices: `min_support = 0.005` and confidence at least 0.30, like the full-year run, and only rules with lift above 1.1 are compared (the same filter I plan for Phase 5, which removes the Reusable bag noise).

## 4.0 Load the dates and split the baskets by season

In [239]:
import json, gc, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

WORK = Path('../data/work').resolve()
products = pd.read_csv('../data/products.csv')
products['Item'] = products['Item'].str.strip()
EN = products.set_index('Item')['Item_EN'].to_dict()

with open(WORK / 'basket_cols.json', encoding='utf-8') as f:
    cols = json.load(f)                          # the 98 product names, same order as the Phase 1 table
col_index = {c: i for i, c in enumerate(cols)}

zip_file = sorted(Path('../data').glob('transactions*.zip'))[0]
print('transactions file:', zip_file.name)
tx = pd.read_csv(zip_file, usecols=['TransactionID', 'Date', 'Item'],
                 dtype={'Item': 'category', 'Date': 'category'})     # category = much less memory

# same name cleaning as Phase 1 (only touches the 98 distinct names)
clean = pd.Index([n.strip() for n in tx['Item'].cat.categories])
new_names = clean.unique()
tx['Item'] = pd.Categorical.from_codes(new_names.get_indexer(clean)[tx['Item'].cat.codes], categories=new_names)

# month of each row (the 365 distinct dates are converted once, not 8 million rows)
dates = pd.to_datetime(pd.Series(tx['Date'].cat.categories))
print('dates from', dates.min().date(), 'to', dates.max().date())
tx['month'] = dates.dt.month.to_numpy()[tx['Date'].cat.codes.to_numpy()]

# check: does each basket have exactly one date?
bm = tx[['TransactionID', 'month']].drop_duplicates()
print(f"baskets: {bm['TransactionID'].nunique():,} | (basket, month) pairs: {len(bm):,}  <- must be equal")
print(bm['month'].value_counts().sort_index().rename('baskets per month').to_frame().T)

transactions file: transactions.zip
dates from 2026-01-01 to 2026-12-31
baskets: 1,000,000 | (basket, month) pairs: 1,000,000  <- must be equal
month                 1      2      3      4      5      6      7      8   \
baskets per month  85180  76964  84754  82145  85202  82231  84997  85246   

month                 9      10     11     12  
baskets per month  81856  84418  82186  84821  


In [240]:
SEASONS = {'summer': [6, 7, 8], 'winter': [12, 1, 2]}

for name, months in SEASONS.items():
    sub = tx[tx['month'].isin(months)]
    row_idx, basket_ids = pd.factorize(sub['TransactionID'])
    col_idx = pd.Categorical(sub['Item'], categories=cols).codes
    assert (col_idx >= 0).all(), 'a product name does not match the Phase 1 list'

    table = np.zeros((len(basket_ids), len(cols)), dtype=bool)
    table[row_idx, col_idx] = True

    d = WORK / name
    d.mkdir(exist_ok=True)
    np.save(d / 'basket.npy', table)
    with open(d / 'basket_cols.json', 'w', encoding='utf-8') as f:
        json.dump(cols, f, ensure_ascii=False)
    print(f'{name}: months {months} -> {table.shape[0]:,} baskets')

tx = None
gc.collect()

summer: months [6, 7, 8] -> 252,474 baskets
winter: months [12, 1, 2] -> 246,965 baskets


66

## 4.1 Mine each season (one separate process per season, same worker as Phase 2)

In [241]:
def mine_season(name, min_support=0.005, max_len=4, min_conf=0.30, timeout_min=30):
    if not (WORK / 'worker.py').exists():
        raise FileNotFoundError('worker.py is missing: run the Phase 2 cell "2.2 The run helper" once (it creates the file).')
    d = WORK / name
    tag = f'{name}_{min_support}'
    out = d / f'run_{tag}.json'
    out.unlink(missing_ok=True)
    print(f'>>> {name}: min_support={min_support}, low_memory=True, max_len={max_len}')
    proc = subprocess.run([sys.executable, str(WORK / 'worker.py'), str(min_support), '1',
                           str(max_len), str(min_conf), tag, str(d)],
                          capture_output=True, text=True, timeout=timeout_min * 60)
    if not out.exists():
        raise RuntimeError('the run failed: ' + proc.stderr[-500:])
    with open(out) as f:
        r = json.load(f)
    print(f"    outcome: {r['outcome']} | itemsets: {r.get('itemsets', 0):,} | rules: {r.get('rules', 0):,} | "
          f"time: {r.get('time_apriori_s', 0):.1f} s | peak memory: {r['peak_mb']:,.0f} MB")
    return tag

tag_summer = mine_season('summer')
tag_winter = mine_season('winter')

>>> summer: min_support=0.005, low_memory=True, max_len=4
    outcome: OK | itemsets: 7,667 | rules: 19,920 | time: 7.4 s | peak memory: 180 MB
>>> winter: min_support=0.005, low_memory=True, max_len=4
    outcome: OK | itemsets: 5,767 | rules: 10,915 | time: 5.7 s | peak memory: 178 MB


## 4.2 The same rules in both seasons

In [242]:
def load_rules(name, tag):
    r = pd.read_pickle(WORK / name / f'rules_{tag}.pkl')
    r = r[r['lift'] > 1.1].copy()                     # same filter as planned for Phase 5
    r['key'] = (r['antecedents'].map(lambda s: ' + '.join(sorted(s))) + ' => '
                + r['consequents'].map(lambda s: ' + '.join(sorted(s))))
    r['if basket has'] = r['antecedents'].map(lambda s: ' + '.join(sorted(EN[i] for i in s)))
    r['then'] = r['consequents'].map(lambda s: ' + '.join(sorted(EN[i] for i in s)))
    return r

rs = load_rules('summer', tag_summer)
rw = load_rules('winter', tag_winter)

keep = ['key', 'if basket has', 'then', 'support', 'confidence', 'lift']
cmp = rs[keep].merge(rw[keep], on=['key', 'if basket has', 'then'], how='outer',
                     suffixes=('_summer', '_winter'), indicator=True)
cmp['present_in'] = cmp['_merge'].map({'both': 'both seasons', 'left_only': 'summer only', 'right_only': 'winter only'})
cmp['simple'] = ~cmp['key'].str.contains(' + ', regex=False)          # one product on each side
simp = cmp[cmp['simple']]

print(f'summer rules (lift > 1.1): {len(rs):,} | winter rules: {len(rw):,}')
print('ALL rules:')
print(cmp['present_in'].value_counts().to_string())
print()
print('SIMPLE rules (one product on each side):')
print(simp['present_in'].value_counts().to_string())

summer rules (lift > 1.1): 16,880 | winter rules: 8,332
ALL rules:
present_in
summer only     11271
both seasons     5609
winter only      2723

SIMPLE rules (one product on each side):
present_in
both seasons    204
winter only      53
summer only      16


In [243]:
show = ['if basket has', 'then', 'support_summer', 'confidence_summer', 'lift_summer',
        'support_winter', 'confidence_winter', 'lift_winter']

both = simp[simp['present_in'] == 'both seasons'].copy()
both['lift ratio (summer / winter)'] = both['lift_summer'] / both['lift_winter']
both['times different'] = both['lift ratio (summer / winter)'].map(lambda x: max(x, 1 / x))
print(f"Simple rules in both seasons: {len(both):,} | median 'times different' in lift: {both['times different'].median():.2f}")
print('The 10 simple rules whose lift changes the most between the seasons:')
both.sort_values('times different', ascending=False).head(10)[show + ['lift ratio (summer / winter)']].round(3).reset_index(drop=True)

Simple rules in both seasons: 204 | median 'times different' in lift: 1.02
The 10 simple rules whose lift changes the most between the seasons:


,if basket has,then,support_summer,confidence_summer,lift_summer,support_winter,confidence_winter,lift_winter,lift ratio (summer / winter)
0,Beef steak,Charcoal,0.187,0.943,3.499,0.025,0.625,12.661,0.276
1,Charcoal,Beef steak,0.187,0.695,3.499,0.025,0.509,12.661,0.276
2,Burger buns,Charcoal,0.177,0.940,3.488,0.024,0.606,12.266,0.284
3,Charcoal,Burger buns,0.177,0.657,3.488,0.024,0.477,12.266,0.284
4,Charcoal,Ketchup,0.157,0.582,3.455,0.021,0.425,11.907,0.290
5,Ketchup,Charcoal,0.157,0.932,3.455,0.021,0.588,11.907,0.290
6,Sausages,Charcoal,0.144,0.927,3.438,0.019,0.572,11.585,0.297
7,Charcoal,Sausages,0.144,0.533,3.438,0.019,0.387,11.585,0.297
8,Beef steak,Burger buns,0.128,0.647,3.432,0.017,0.429,11.045,0.311
9,Burger buns,Beef steak,0.128,0.682,3.432,0.017,0.444,11.045,0.311


In [244]:
print('SUMMER ONLY: top 10 simple rules by lift (not found in winter)')
simp[simp['present_in'] == 'summer only'].sort_values('lift_summer', ascending=False).head(10)[show].round(3).reset_index(drop=True)

SUMMER ONLY: top 10 simple rules by lift (not found in winter)


,if basket has,then,support_summer,confidence_summer,lift_summer,support_winter,confidence_winter,lift_winter
0,Charcoal,Relish,0.092,0.342,3.295,NaN,NaN,NaN
1,Sausages,BBQ sauce,0.070,0.453,3.292,NaN,NaN,NaN
2,Beef steak,Relish,0.067,0.337,3.249,NaN,NaN,NaN
3,Burger buns,Relish,0.063,0.336,3.240,NaN,NaN,NaN
4,Relish,Ketchup,0.056,0.541,3.209,NaN,NaN,NaN
5,Ketchup,Relish,0.056,0.333,3.209,NaN,NaN,NaN
6,Sausages,Relish,0.051,0.331,3.187,NaN,NaN,NaN
7,Relish,Sausages,0.051,0.494,3.187,NaN,NaN,NaN
8,Relish,BBQ sauce,0.045,0.435,3.160,NaN,NaN,NaN
9,BBQ sauce,Relish,0.045,0.328,3.160,NaN,NaN,NaN


In [245]:
print('WINTER ONLY: top 10 simple rules by lift (not found in summer)')
simp[simp['present_in'] == 'winter only'].sort_values('lift_winter', ascending=False).head(10)[show].round(3).reset_index(drop=True)

WINTER ONLY: top 10 simple rules by lift (not found in summer)


,if basket has,then,support_summer,confidence_summer,lift_summer,support_winter,confidence_winter,lift_winter
0,Raclette cheese,Fondue broth,NaN,NaN,NaN,0.067,0.552,6.808
1,Raclette cheese,Deli meats,NaN,NaN,NaN,0.064,0.525,6.718
2,Raclette cheese,Pickles,NaN,NaN,NaN,0.059,0.486,6.679
3,Raclette cheese,Rustic baguette,NaN,NaN,NaN,0.055,0.453,6.538
4,Rustic baguette,Raclette cheese,NaN,NaN,NaN,0.055,0.797,6.538
5,White wine,Raclette cheese,NaN,NaN,NaN,0.051,0.787,6.456
6,Raclette cheese,White wine,NaN,NaN,NaN,0.051,0.416,6.456
7,Deli meats,Fondue broth,NaN,NaN,NaN,0.040,0.513,6.333
8,Fondue broth,Deli meats,NaN,NaN,NaN,0.040,0.495,6.333
9,Raclette cheese,Potatoes,NaN,NaN,NaN,0.045,0.368,6.319


**Reading the comparison.**
- Most simple rules are stable across seasons: 204 of 273 appear in both, and their lift changes by only 1.02 times (median). Year-round habits like sushi (Nori seaweed with Sushi rice) do not depend on the season.
- Summer has about twice as many rules as winter (16,880 vs 8,332 with lift above 1.1), because the BBQ products are bought together in many combinations.
- The rules whose lift changes the most are the BBQ ones (Beef steak, Charcoal, Burger buns, Ketchup, Sausages). Their lift is about 3.5 in summer and 11 to 13 in winter, but the support falls from about 18% of baskets in summer to about 2% in winter. The lift goes up because few people buy BBQ products in winter, so those who do are unusual and buy the whole kit. Lift must be read together with support.
- The 53 winter-only simple rules are mostly the raclette group (Raclette cheese with Fondue broth, Deli meats, Pickles and Rustic baguette; lift about 6.5).

## 4.3 Does a rule really disappear? Measure it directly
"Only in summer" could just mean "slightly under my thresholds in winter". So for the strongest one-season simple rules, I measure the rule directly in each season's baskets, and in the whole year (what a year-round system would use).

In [246]:
X = {'year': np.load(WORK / 'basket.npy'),
     'summer': np.load(WORK / 'summer' / 'basket.npy'),
     'winter': np.load(WORK / 'winter' / 'basket.npy')}

def rule_stats(table, ant, cons):
    """support, confidence, lift of 'ant => cons' measured directly on a basket table"""
    a = table[:, [col_index[i] for i in ant]].all(axis=1)
    c = table[:, [col_index[i] for i in cons]].all(axis=1)
    p_a, p_c, p_ac = a.mean(), c.mean(), (a & c).mean()
    if p_a == 0 or p_c == 0:
        return p_ac, np.nan, np.nan
    conf = p_ac / p_a
    return p_ac, conf, conf / p_c

def fmt(s, c, l):
    return f'{s:.1%} / {c:.0%} / {l:.1f}x' if c == c else f'{s:.1%} / - / -'

def compare_rules(keys):
    rows = []
    for key in keys:
        ant, cons = [part.split(' + ') for part in key.split(' => ')]
        row = {'if basket has': ' + '.join(EN[i] for i in ant), 'then': ' + '.join(EN[i] for i in cons)}
        for name, label in [('summer', 'summer'), ('winter', 'winter'), ('year', 'whole year')]:
            row[label] = fmt(*rule_stats(X[name], ant, cons))
        rows.append(row)
    return pd.DataFrame(rows)

top_summer = simp[simp['present_in'] == 'summer only'].sort_values('lift_summer', ascending=False).head(5)['key']
top_winter = simp[simp['present_in'] == 'winter only'].sort_values('lift_winter', ascending=False).head(5)['key']

print('Each cell reads: support / confidence / lift')
compare_rules(list(top_summer) + list(top_winter))

Each cell reads: support / confidence / lift


,if basket has,then,summer,winter,whole year
0,Charcoal,Relish,9.2% / 34% / 3.3x,1.2% / 25% / 9.3x,4.5% / 32% / 5.5x
1,Sausages,BBQ sauce,7.0% / 45% / 3.3x,1.0% / 29% / 9.2x,3.5% / 41% / 5.5x
2,Beef steak,Relish,6.7% / 34% / 3.2x,0.9% / 22% / 8.3x,3.3% / 31% / 5.3x
3,Burger buns,Relish,6.3% / 34% / 3.2x,0.8% / 22% / 8.1x,3.1% / 31% / 5.3x
4,Relish,Ketchup,5.6% / 54% / 3.2x,0.8% / 28% / 7.8x,2.8% / 47% / 5.2x
5,Raclette cheese,Fondue broth,0.8% / 28% / 11.9x,6.7% / 55% / 6.8x,3.4% / 49% / 10.2x
6,Raclette cheese,Deli meats,0.7% / 27% / 12.0x,6.4% / 53% / 6.7x,3.2% / 47% / 10.0x
7,Raclette cheese,Pickles,0.7% / 25% / 11.3x,5.9% / 49% / 6.7x,3.0% / 43% / 9.8x
8,Raclette cheese,Rustic baguette,0.6% / 24% / 11.0x,5.5% / 45% / 6.5x,2.8% / 40% / 9.6x
9,Rustic baguette,Raclette cheese,0.6% / 30% / 11.0x,5.5% / 80% / 6.5x,2.8% / 65% / 9.6x


## 4.3b Does any one-season rule disappear completely?
For every simple rule found in only one season, measure it directly in the other season (support, confidence, lift). A rule has really disappeared if its lift there is about 1 (no link left), or if almost nobody buys the products together.

In [247]:
one_season = simp[simp['present_in'] != 'both seasons']

rows = []
for key, where in zip(one_season['key'], one_season['present_in']):
    ant, cons = [part.split(' + ') for part in key.split(' => ')]
    mined_in, other = ('summer', 'winter') if where == 'summer only' else ('winter', 'summer')
    s, c, l = rule_stats(X[other], ant, cons)
    rows.append({'if basket has': ' + '.join(EN[i] for i in ant), 'then': ' + '.join(EN[i] for i in cons),
                 'mined in': mined_in, 'support there': s, 'confidence there': c, 'lift there': l})
gone = pd.DataFrame(rows)

print(f'{len(gone)} simple rules were found in only one season. Measured in the OTHER season:')
print(f"  lowest support : {gone['support there'].min():.2%}")
print(f"  lowest lift    : {gone['lift there'].min():.2f}")
print(f"  rules with lift <= 1.1 (no link left): {(gone['lift there'].isna() | (gone['lift there'] <= 1.1)).sum()}")
print(f"  rules with support under 1%          : {(gone['support there'] < 0.01).sum()}")
print(f"  rules with support under 0.5%        : {(gone['support there'] < 0.005).sum()}")
print()
print('The 10 rules with the weakest link in the other season:')
gone.sort_values('lift there').head(10).round(3).reset_index(drop=True)

69 simple rules were found in only one season. Measured in the OTHER season:
  lowest support : 0.26%
  lowest lift    : 1.50
  rules with lift <= 1.1 (no link left): 0
  rules with support under 1%          : 61
  rules with support under 0.5%        : 36

The 10 rules with the weakest link in the other season:


,if basket has,then,mined in,support there,confidence there,lift there
0,Popcorn,Beer,winter,0.006,0.275,1.504
1,Pretzels,Beer,winter,0.006,0.291,1.591
2,Beer,Potato chips,winter,0.012,0.067,2.083
3,Beer,BBQ sauce,summer,0.011,0.076,2.361
4,Beer,Sausages,summer,0.012,0.082,2.462
5,Beer,Ketchup,summer,0.013,0.090,2.525
6,Beer,Burger buns,summer,0.014,0.100,2.561
7,Beer,Beef steak,summer,0.015,0.106,2.632
8,Beer,Charcoal,summer,0.020,0.139,2.809
9,Popcorn,Salsa,winter,0.003,0.126,5.583


**Does a rule disappear completely? No.**

I measured the 69 simple rules found in only one season directly in the other season. None of them disappears:
- the lowest lift in the other season is 1.50, and no rule has a lift of 1.1 or less (0 of 69);
- the lowest support is 0.26% of baskets: 61 of the 69 rules are under 1% of baskets in the other season, and 36 are under 0.5%.

What happens instead: the rule falls out of the mined list because its support or its confidence drops under my thresholds. Example: Charcoal → Relish has a confidence of 34% in summer and 25% in winter, under my limit of 30%, so it is missing from the winter rules. But the link is still there (lift 9.3 in winter). The real seasonal effect is the **volume**: Charcoal → Relish is in 9.2% of summer baskets and 1.2% of winter baskets, and Raclette cheese → Fondue broth is in 6.7% of winter baskets and 0.8% of summer baskets. That is about 8 times fewer baskets.

The brief asks for a rule that disappears entirely. I did not find one in this data. I report it as a negative result. The closest ones are Popcorn → Beer and Pretzels → Beer, which were mined in winter only: in summer their lift falls to about 1.5 and they are in only 0.6% of baskets.

## 4.4 (replaces the earlier 4.4) What happens if the website serves year-round rules in January?
A year-round rule states one confidence and one lift for the whole year. Here I take every simple year-round rule and compare what the website would say (whole-year numbers) with what is true in each season.

In [248]:
yr = pd.read_pickle(WORK / 'rules_0.005_low_len4.pkl')
yr = yr[(yr['lift'] > 1.1) & (yr['antecedents'].map(len) == 1) & (yr['consequents'].map(len) == 1)].copy()

rows = []
for ant, cons, conf_y, lift_y, sup_y in zip(yr['antecedents'], yr['consequents'], yr['confidence'], yr['lift'], yr['support']):
    row = {'if basket has': ' + '.join(EN[i] for i in ant), 'then': ' + '.join(EN[i] for i in cons),
           'year confidence': conf_y, 'year lift': lift_y, 'year support': sup_y}
    for name in ['summer', 'winter']:
        s, c, l = rule_stats(X[name], ant, cons)
        row[f'{name} support'], row[f'{name} confidence'], row[f'{name} lift'] = s, c, l
    rows.append(row)
yy = pd.DataFrame(rows)

print(f'{len(yy)} simple year-round rules (lift > 1.1)')
for name in ['summer', 'winter']:
    conf_gap = (yy[f'{name} confidence'] / yy['year confidence'] - 1).abs()
    sup_ratio = yy[f'{name} support'] / yy['year support']
    print(f'In {name}:')
    print(f'  confidence more than 20% away from the whole-year figure: {(conf_gap > 0.2).sum()} rules ({(conf_gap > 0.2).mean():.0%})')
    print(f'  fewer than half as many baskets as the whole-year support : {(sup_ratio < 0.5).sum()} rules ({(sup_ratio < 0.5).mean():.0%})')

yy['winter / year confidence'] = yy['winter confidence'] / yy['year confidence']
yy['gap'] = (yy['winter / year confidence'] - 1).abs()
print()
print('January: the 10 rules where the whole-year confidence is most wrong')
out = yy.sort_values('gap', ascending=False).head(10)[['if basket has', 'then', 'year confidence', 'winter confidence',
                                                      'year lift', 'winter lift', 'year support', 'winter support']]
out.round(3).reset_index(drop=True)

262 simple year-round rules (lift > 1.1)
In summer:
  confidence more than 20% away from the whole-year figure: 54 rules (21%)
  fewer than half as many baskets as the whole-year support : 51 rules (19%)
In winter:
  confidence more than 20% away from the whole-year figure: 72 rules (27%)
  fewer than half as many baskets as the whole-year support : 51 rules (19%)

January: the 10 rules where the whole-year confidence is most wrong


,if basket has,then,year confidence,winter confidence,year lift,winter lift,year support,winter support
0,Beer,Potato chips,0.301,0.613,3.294,3.651,0.045,0.089
1,Beer,Charcoal,0.454,0.139,3.228,2.809,0.068,0.020
2,Beer,Burger buns,0.314,0.100,3.105,2.561,0.047,0.014
3,Beer,Beef steak,0.331,0.106,3.139,2.632,0.050,0.015
4,Relish,Sausages,0.431,0.253,5.116,7.563,0.025,0.007
5,Relish,Burger buns,0.531,0.314,5.258,8.075,0.031,0.008
6,Relish,Ketchup,0.469,0.278,5.159,7.793,0.028,0.008
7,Relish,Charcoal,0.770,0.459,5.475,9.294,0.045,0.012
8,Relish,Beef steak,0.559,0.334,5.295,8.313,0.033,0.009
9,Relish,BBQ sauce,0.377,0.233,4.991,7.257,0.022,0.006


**Serving year-round rules in January: what goes wrong.**

A year-round rule gives one confidence for the whole year. I compared it with the real confidence in each season for 262 simple rules:
- in winter, 72 rules (27%) have a confidence more than 20% away from the whole-year figure (in summer: 54 rules, 21%);
- in each season, 51 rules (19%) apply to fewer than half as many baskets as the year-round figure suggests.

Examples in January (year-round figure → real winter figure):
- Beer → Charcoal: 45% → 14%. Beer → Burger buns: 31% → 10%. Beer → Beef steak: 33% → 11%.
- Relish → Charcoal: 77% → 46%. Relish → Ketchup: 47% → 28%.
- The other way around, Beer → Potato chips: 30% → 61%. The website would understate it by half.

So in January the website would show "customers who buy Beer also buy Charcoal 45% of the time" when it is 14%. It would also give BBQ suggestions a strong score, while far fewer customers buy BBQ products in winter, and it would not rank Beer → Potato chips as high as it deserves (61% in winter against 30% for the whole year). The explanation shown to the customer would be wrong, and the explanation has to be true to meet the explainability requirement (Law 25).

**How often to re-mine.** Mining is cheap here: one season (about 250,000 baskets) takes 6 to 7 seconds and under 180 MB, and the whole year takes about 30 to 40 seconds in most runs (one repeat took 222 s) and 410 MB with `low_memory=True`. So the cost is not the limit. My conclusion:
- re-mine at least at every change of season (4 times a year), and keep one rule set per season (summer, winter, and spring/autumn from the last three months), chosen by the date;
- better, re-mine every month automatically with the most recent months, because new products, promotions and trends also change the rules;
- keep a year-round rule set only as a fallback.

Limit: I have only one year of data, so I cannot tell whether these patterns repeat every year or were one-time events.

---
# Phase 5 — Save the rules (offline), so the app only loads a file

Mining is expensive and done offline here. The Streamlit app never mines: it only reads `models/rules.json` and `models/metadata.json`.

Choices (to write in the README):
- Rules come from the **full-year run**: `min_support = 0.005`, confidence at least 0.30, itemsets of at most 4 products.
- Only rules with **lift above 1.1** are kept. Lift above 1 is the required filter. Every rule with a lift between 1 and 1.1 has the Reusable bag on the "then" side (section 3.6), so 1.1 removes only noise.
- Only rules with **one product on the right side** are kept, because the app suggests one product at a time.
- **Redundant rules are removed.** A product with no link to the rest (Cookies, for example) can be added to the left side of a good rule without changing anything: "Charcoal → Beef steak" and "Charcoal + Cookies → Beef steak" have nearly the same confidence and lift. If both were kept, the app could explain a suggestion with a product that has nothing to do with it. So a rule with more products on the left is kept only if its confidence beats every simpler version of the same rule by at least 2 points.

## 5.1 Filter the rules

In [249]:
import json, datetime
import importlib.metadata as metadata
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path('../data')
WORK = DATA / 'work'
MODELS = Path('../models')
MODELS.mkdir(exist_ok=True)

MIN_SUPPORT, MIN_CONFIDENCE, MIN_LIFT, MAX_LEN = 0.005, 0.30, 1.1, 4
MIN_IMPROVEMENT = 0.02          # a longer rule must beat its simpler versions by 2 points of confidence (see the text above)
RULES_FILE = WORK / 'rules_0.005_low_len4.pkl'

rules = pd.read_pickle(RULES_FILE)

# 1) lift above 1.1, 2) one product on the right side
after_lift = rules[rules['lift'] > MIN_LIFT]
single = after_lift[after_lift['consequents'].map(len) == 1]

# 3) remove redundant rules. First: the confidence of every mined rule with one product on the right
all_single = rules[rules['consequents'].map(len) == 1]
conf_of = {(a, next(iter(c))): cf
           for a, c, cf in zip(all_single['antecedents'], all_single['consequents'], all_single['confidence'])}

def best_simpler_confidence(ant, cons):
    """highest confidence among the simpler versions of a rule (same rule, fewer products on the left)"""
    best = 0.0
    for k in range(1, len(ant)):
        for sub in combinations(sorted(ant), k):
            best = max(best, conf_of.get((frozenset(sub), cons), 0.0))
    return best

keep = [cf - best_simpler_confidence(a, next(iter(c))) >= MIN_IMPROVEMENT
        for a, c, cf in zip(single['antecedents'], single['consequents'], single['confidence'])]
final = single[keep].sort_values('lift', ascending=False)

print(f'rules mined (min_support {MIN_SUPPORT}, confidence >= {MIN_CONFIDENCE}) : {len(rules):,}')
print(f'removed because lift <= {MIN_LIFT}                          : {len(rules) - len(after_lift):,}')
print(f'removed because of more than one product on the right : {len(after_lift) - len(single):,}')
print(f'removed because redundant (extra products add nothing) : {len(single) - len(final):,}')
print(f'rules kept for the app                                : {len(final):,}')

rules mined (min_support 0.005, confidence >= 0.3) : 13,892
removed because lift <= 1.1                          : 2,805
removed because of more than one product on the right : 4,368
removed because redundant (extra products add nothing) : 5,755
rules kept for the app                                : 964


## 5.2 Save `models/rules.json`

In [250]:
rules_out = [{'if': sorted(a), 'then': next(iter(c)),
              'support': round(float(s), 6), 'confidence': round(float(cf), 6), 'lift': round(float(l), 6)}
             for a, c, s, cf, l in zip(final['antecedents'], final['consequents'],
                                       final['support'], final['confidence'], final['lift'])]

with open(MODELS / 'rules.json', 'w', encoding='utf-8') as f:        # one rule per line, easy to read
    f.write('[\n' + ',\n'.join(json.dumps(r, ensure_ascii=False) for r in rules_out) + '\n]\n')

print(f"models/rules.json saved: {len(rules_out):,} rules, {(MODELS / 'rules.json').stat().st_size / 1e6:.1f} MB")
print('Strongest rule:', rules_out[0])

models/rules.json saved: 964 rules, 0.1 MB
Strongest rule: {'if': ['Charcuteries', 'Pain baguette rustique', 'Pommes de terre'], 'then': 'Fromage à raclette', 'support': 0.006642, 'confidence': 0.999699, 'lift': 14.592441}


## 5.3 Coverage and `models/metadata.json`

In [251]:
with open(WORK / 'basket_cols.json', encoding='utf-8') as f:
    cols = json.load(f)
X = np.load(WORK / 'basket.npy')
item_support = {c: round(float(s), 6) for c, s in zip(cols, X.mean(axis=0))}     # share of baskets, per product

products = pd.read_csv(DATA / 'products.csv')
products['Item'] = products['Item'].str.strip()
EN = products.set_index('Item')['Item_EN'].to_dict()

recommendable = {r['then'] for r in rules_out}                                   # products that can be suggested
in_any_rule = recommendable | {i for r in rules_out for i in r['if']}            # products in at least one rule
never = sorted(set(cols) - recommendable, key=lambda i: item_support[i])

metadata_out = {
    'mining_date': datetime.datetime.fromtimestamp(RULES_FILE.stat().st_mtime).date().isoformat(),
    'algorithm': 'mlxtend apriori (low_memory=True) + association_rules',
    'mlxtend_version': metadata.version('mlxtend'),
    'data': {'baskets': int(X.shape[0]), 'products': len(cols)},
    'thresholds': {'min_support': MIN_SUPPORT, 'min_confidence': MIN_CONFIDENCE, 'min_lift': MIN_LIFT,
                   'max_itemset_length': MAX_LEN, 'products_on_the_right_side': 1,
                   'min_confidence_improvement_over_simpler_rule': MIN_IMPROVEMENT},
    'rules': {'mined': int(len(rules)), 'kept': len(rules_out)},
    'coverage': {'recommendable_products': len(recommendable), 'total_products': len(cols),
                 'products_in_at_least_one_rule': len(in_any_rule),
                 'never_recommended': never},
    'item_support': item_support,
}
with open(MODELS / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata_out, f, ensure_ascii=False, indent=1)

print(f'COVERAGE: the system can recommend {len(recommendable)} of {len(cols)} products '
      f'({len(recommendable) / len(cols):.0%}).')
print(f'Products that appear in no rule at all: {len(cols) - len(in_any_rule)}')
print()
print('Products that can NEVER be recommended (rarest first):')
for i in never:
    print(f'  {EN[i]:30} bought in {item_support[i]:.2%} of baskets')

COVERAGE: the system can recommend 57 of 98 products (58%).
Products that appear in no rule at all: 40

Products that can NEVER be recommended (rarest first):
  Foie gras                      bought in 0.08% of baskets
  Aged balsamic vinegar          bought in 0.08% of baskets
  Caviar                         bought in 0.08% of baskets
  Truffle oil                    bought in 0.08% of baskets
  Saffron                        bought in 0.08% of baskets
  Carrots                        bought in 3.14% of baskets
  Strawberry jam                 bought in 5.54% of baskets
  Chicken breast                 bought in 5.59% of baskets
  Ice cider                      bought in 5.82% of baskets
  Paper towels                   bought in 6.00% of baskets
  Bagels                         bought in 6.27% of baskets
  Baguette                       bought in 6.94% of baskets
  Dark chocolate                 bought in 7.27% of baskets
  Frozen peas                    bought in 7.28% of baskets
 

## 5.4 Look at the rules that were saved

In [252]:
kept = pd.DataFrame(rules_out)
kept['n_if'] = kept['if'].map(len)
print('Rules by number of products on the left:')
print(kept['n_if'].value_counts().sort_index().rename('rules').to_frame().T.to_string())
print()
print('Lift of the kept rules:')
print(kept['lift'].describe().round(2).to_string())
print()
n_per_product = kept.groupby('then').size().sort_values(ascending=False)
print('Products suggested by the most rules:')
print(pd.Series(n_per_product.head(5).values, index=[EN[i] for i in n_per_product.head(5).index]).to_string())

Rules by number of products on the left:
n_if     1    2   3
rules  262  682  20

Lift of the kept rules:
count    964.00
mean       8.05
std        2.66
min        2.14
25%        6.35
50%        7.41
75%       10.05
max       14.59

Products suggested by the most rules:
Beer               52
Raclette cheese    29
Charcoal           28
Eggs               28
Burger buns        28


## 5.5 Check that the app can read the files, and write `requirements.txt`

In [253]:
# the app must be able to load exactly what was saved
with open(MODELS / 'rules.json', encoding='utf-8') as f:
    reloaded = json.load(f)
with open(MODELS / 'metadata.json', encoding='utf-8') as f:
    meta_reloaded = json.load(f)
assert len(reloaded) == meta_reloaded['rules']['kept'] == len(rules_out)
print(f"OK: rules.json ({len(reloaded):,} rules) and metadata.json (mined on {meta_reloaded['mining_date']}) can be read.")

# requirements.txt with the exact versions used here
lines = []
for pkg in ['pandas', 'numpy', 'mlxtend', 'psutil', 'streamlit']:
    try:
        lines.append(f'{pkg}=={metadata.version(pkg)}')
    except metadata.PackageNotFoundError:
        lines.append(f'{pkg}   # not installed here: run "pip install {pkg}" and run this cell again')
with open('../requirements.txt', 'w') as f:
    f.write('\n'.join(lines) + '\n')
print()
print(open('../requirements.txt').read())

OK: rules.json (964 rules) and metadata.json (mined on 2026-11-01) can be read.

pandas==3.0.5
numpy==2.5.2
mlxtend==0.25.0
psutil==7.2.2
streamlit==1.62.0



---
# Phase 6 — Coverage and the famous rule

## 6.1 Why can some products never be recommended?

In [254]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

DATA, MODELS = Path('../data'), Path('../models')
with open(MODELS / 'rules.json', encoding='utf-8') as f:
    rules_json = json.load(f)
with open(MODELS / 'metadata.json', encoding='utf-8') as f:
    meta = json.load(f)

products = pd.read_csv(DATA / 'products.csv')
products['Item'] = products['Item'].str.strip()
EN = products.set_index('Item')['Item_EN'].to_dict()
CAT = products.set_index('Item')['Category'].to_dict()
share = meta['item_support']
min_support = meta['thresholds']['min_support']

right = {r['then'] for r in rules_json}
left = {i for r in rules_json for i in r['if']}
never = [i for i in share if i not in right]

too_rare = [i for i in never if share[i] < min_support]
no_rule = [i for i in never if share[i] >= min_support and i not in left]
only_left = [i for i in never if share[i] >= min_support and i in left]

print(f"Can be recommended         : {len(right)} of {len(share)} products ({len(right) / len(share):.0%})")
print(f"Never recommended          : {len(never)}")
print(f"  too rare (under {min_support:.1%} of baskets)       : {len(too_rare)}  -> {[EN[i] for i in too_rare]}")
print(f"  common, but in no rule at all       : {len(no_rule)}  -> {[EN[i] for i in no_rule]}")
print(f"  common, only on the left of a rule  : {len(only_left)}  (they can trigger a suggestion, but are never suggested)")
print()
print('Categories of the products that are never recommended:')
pd.Series([CAT[i] for i in never]).value_counts().rename('products').to_frame().T

Can be recommended         : 57 of 98 products (58%)
Never recommended          : 41
  too rare (under 0.5% of baskets)       : 5  -> ['Caviar', 'Foie gras', 'Truffle oil', 'Saffron', 'Aged balsamic vinegar']
  common, but in no rule at all       : 35  -> ['Bagels', 'Baguette', 'Butter', 'Peanut butter', 'Cookies', 'Carrots', 'Dark chocolate', 'Ice cider', 'Strawberry jam', 'Shrimp', 'Ice cream', 'Cereal', 'Laundry detergent', 'Sparkling water', 'Cheddar cheese', 'Swiss cheese', '2% milk', 'Dijon mustard', 'Mixed nuts', 'Onions', 'Sliced bread', 'Paper towels', 'Toilet paper', 'Frozen peas', 'Frozen pizza', 'Chicken breast', 'Apples', 'Basmati rice', 'Garbage bags', 'Dish soap', 'White sugar', 'Canned tuna', 'Green tea', 'Tortillas', 'Greek yogurt']
  common, only on the left of a rule  : 1  (they can trigger a suggestion, but are never suggested)

Categories of the products that are never recommended:


,Épicerie,Maison,Spécialités,Boulangerie,Produits laitiers,Collations,Fruits et légumes,Boissons,Surgelés,Poissons,Fromages,Viandes
products,6,6,5,4,3,3,3,3,3,2,2,1


## 6.2 The famous example: Diapers and Beer

In [255]:
WORK = DATA / 'work'
with open(WORK / 'basket_cols.json', encoding='utf-8') as f:
    cols = json.load(f)
X = np.load(WORK / 'basket.npy')
idx = {c: i for i, c in enumerate(cols)}
A, B = 'Couches', 'Bière'                     # Diapers, Beer

a, b = X[:, idx[A]], X[:, idx[B]]
p_a, p_b, p_ab = a.mean(), b.mean(), (a & b).mean()
print(f'Diapers are in {p_a:.2%} of baskets, Beer in {p_b:.2%}, both together in {p_ab:.2%}')
print(f'  Diapers -> Beer : confidence {p_ab / p_a:.1%} | lift {p_ab / p_a / p_b:.2f}')
print(f'  Beer -> Diapers : confidence {p_ab / p_b:.1%} | lift {p_ab / p_b / p_a:.2f}')
print()

pair = [r for r in rules_json if {A, B} <= (set(r['if']) | {r['then']})]
print(f'Rules in rules.json that contain both Diapers and Beer: {len(pair)}')
pd.DataFrame([{'if': ' + '.join(EN[i] for i in r['if']), 'then': EN[r['then']], 'support': f"{r['support']:.2%}",
               'confidence': f"{r['confidence']:.1%}", 'lift': round(r['lift'], 2)} for r in pair[:10]])

Diapers are in 8.40% of baskets, Beer in 15.05%, both together in 3.07%
  Diapers -> Beer : confidence 36.6% | lift 2.43
  Beer -> Diapers : confidence 20.4% | lift 2.43

Rules in rules.json that contain both Diapers and Beer: 10


,if,then,support,confidence,lift
0,Beer + Baby wipes,Diapers,2.29%,92.2%,10.97
1,Beer + Baby food,Diapers,1.68%,89.5%,10.66
2,Beer + Diapers,Baby wipes,2.29%,74.7%,10.64
3,Beer + Baby formula,Diapers,1.24%,86.8%,10.33
4,Beer + Diapers,Baby food,1.68%,54.6%,9.87
5,Beer + Diapers,Baby formula,1.24%,40.3%,9.10
6,Diapers + Baby formula,Beer,1.24%,41.0%,2.73
7,Diapers + Baby wipes,Beer,2.29%,41.0%,2.72
8,Diapers + Baby food,Beer,1.68%,41.0%,2.72
9,Diapers,Beer,3.07%,36.6%,2.43
